In [ ]:
import sys
sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm

In [ ]:
# m, fuseMarkers = inflation.triangulate_channel_walls(*parametric_pillows.squareWithVerticalChannels(0.9, 8), 0.001)
m, fuseMarkers = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(8, 50), 0.001)
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=768, height=640)
viewer.showWireframe()
viewer.show()

In [ ]:
import time
inflation.benchmark_reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
niter = 5000
iterations_per_output = 10
opts.niter = iterations_per_output
isheet.pressure = 30
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.01) # Allow some mesh synchronization time for pythreejs
    if cr.numIters() < iterations_per_output: break
    #isheet.writeDebugMesh('concentric_circles/inflation_tf_step_{}.msh'.format(step))
inflation.benchmark_report()

In [ ]:
isa = inflation.InflatedSurfaceAnalysis(isheet)
opt = inflation.SheetOptimizer(isheet, isa.inflatedSurface())

In [ ]:
xorig = opt.getVars()

In [ ]:
# Note: the fitting term has a discontinuous gradient at the initial configuration where the
# wall vertices coincide with the vertices of the target surface mesh that was constructed
# from them; perturb away from this configuration.
opt.setVars(xorig + 1e-5 * np.random.uniform(low=-1, high=1, size=xorig.shape))

In [ ]:
# We need a larger magnitude perturbation to activate the collapse barrier term.
# (Keep running this until a finite, nonzero energy is printed).
opt.setVars(xorig + 4e-3 * np.random.uniform(low=-1, high=1, size=xorig.shape))
opt.energy(opt.EnergyType.CollapseBarrier)

In [ ]:
fd_validation.validateGrad(opt, fd_eps=1e-7, etype=opt.EnergyType.CollapseBarrier)

In [ ]:
fd_validation.validateGrad(opt, fd_eps=1e-7, etype=opt.EnergyType.Fitting)

In [ ]:
from numpy.linalg import norm
etype = opt.EnergyType.Full
err, fd_delta_grad, an_delta_grad = fd_validation.validateHessian(opt, fd_eps=1e-8, etype=etype)
# Test only the equilibrium rows of the Hessian
# an_delta_grad[opt.numEquilibriumVars():] = 0.0
# fd_delta_grad[opt.numEquilibriumVars():] = 0.0
print(norm(an_delta_grad - fd_delta_grad) / norm(fd_delta_grad))
print(an_delta_grad[an_delta_grad != 0])
print(fd_delta_grad[an_delta_grad != 0])

In [ ]:
tsf = opt.targetSurfaceFitter()

In [ ]:
deviations = np.unique(np.where(tsf.gradient() != 0)[0])
tsf.closestSurfPts[deviations, :]

In [ ]:
import vis, mesh
cubeMesh = mesh.Mesh(*vis.primitives.cubes(tsf.closestSurfPts[deviations, :], 0.005))

In [ ]:
tsf.queryPoints[deviations, :]

In [ ]:
tsf.gradient()[np.unique(np.where(tsf.gradient() != 0)[0]), :]

In [ ]:
import sparsity_analysis
sparsity_analysis.spy(opt.hessianSparsityPattern(1.0))